# **Regression: Ordinary Least Squares (OLS) Linear Regression**

## **Justification of Preprocessing Strategy**

### **Scale Invariance vs. Coefficient Interpretation**
Ordinary Least Squares (OLS) **Linear Regression** is mathematically scale-invariant regarding its final predictions. This means that whether the data is raw, standardized, or normalized, the model will output the exact same predictions. However, the magnitude and scale of the learned coefficients ($\beta$ weights) depend entirely on the scale of their corresponding features. To ensure that we can directly compare the clinical importance of each feature on the `diabetes_risk_score` (interpreting which variable has the strongest impact), we will evaluate both **Standardization** and **Normalization** to find the most numerically stable representation.

### **Data Integrity and Leakage Prevention**
To successfully shift our objective from classification to regression, we strictly drop the previous classification targets (`diagnosed_diabetes` and `diabetes_stage`) to avoid any data leakage. Furthermore, because the target variable `diabetes_risk_score` is continuous, we **omit the stratification parameter** during the train-test split, as stratification is mathematically exclusive to categorical classes.


## **Experiment Design**

We have designed a tournament of **2 focused runs** to establish our linear baseline environment, evaluating performance using **MAE, RMSE, and $R^2$**:

* **Standardized OLS Linear Regression**: Training the classical linear model on features processed via `StandardScaler` to evaluate performance under a normally distributed feature space.
* **Normalized OLS Linear Regression**: Training the classical linear model on features processed via `MinMaxScaler` to evaluate performance under a strictly bounded [0, 1] feature space.


In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Linear_Regression")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr, y_tr_pred))
    mlflow.log_metric("rmse_train", np.sqrt(mean_squared_error(y_tr, y_tr_pred)))
    mlflow.log_metric("r2_train", r2_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("mae_test", mean_absolute_error(y_te, y_te_pred))
    mlflow.log_metric("rmse_test", np.sqrt(mean_squared_error(y_te, y_te_pred)))
    mlflow.log_metric("r2_test", r2_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# OLS Tournament Loop
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"OLS_LinearReg_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        model = LinearRegression()
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("model_type", "OLS_LinearRegression")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("fit_intercept", model.fit_intercept)
        
        log_regression_metrics(model, X_train_scaled, y_train, X_test_scaled, y_test, duration)


## **Winner Run Selection (Priority Elimination Framework)**

### **Selection Criteria (in priority order)**
1. **Priority 1 (60% weight): Lowest MAE (Test)** — Clinical proximity; minimizes average day-to-day prediction error on unseen data.
2. **Priority 2 (30% weight): RMSE proportional to MAE** — Rejects runs where RMSE spikes relative to MAE, indicating catastrophic errors.
3. **Priority 3 (10% weight): Acceptable R² (Test)** — Confirms statistical fit quality on unseen data.
4. **Tiebreaker: Lowest Fit Time** — Applied only if a technical tie exists in MAE, RMSE, and R² metrics.

### **All Runs: Summary Table with Train and Test Metrics**

| Run | Scaler | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time (s) |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| **OLS_LinearReg_Standardization** | **Standardization** | **0.39666** | **0.40386** | **0.69367** | **0.71129** | **0.99413** | **0.99387** | **0.20648** |
| OLS_LinearReg_Normalization | Normalization | 0.39666 | 0.40386 | 0.69367 | 0.71129 | 0.99413 | 0.99387 | 0.19760 |

### **Step-by-Step Elimination Process**

**Step 1: Filter by Lowest Test MAE (Priority 1 — 60%)**
- Threshold: Test MAE ≤ 0.40386
- Candidates passing: OLS_LinearReg_Standardization (0.40386), OLS_LinearReg_Normalization (0.40386)
- Status: Both runs achieve identical test MAE

**Step 2: Verify RMSE Proportional to MAE (Priority 2 — 30%)**
- Both candidates (Test): RMSE = 0.71129 (identical)
- MAE-to-RMSE ratio: 0.40386 / 0.71129 ≈ 0.568 (same for both)
- Status: **No catastrophic divergence detected**. Both candidates pass.

**Step 3: Confirm Acceptable Test R² (Priority 3 — 10%)**
- Both candidates: R² (Test) = 0.99387 (identical, excellent fit)
- Status: Both candidates confirmed acceptable.

**Step 4: Apply Tiebreaker — Lowest Fit Time**
- OLS_LinearReg_Normalization: 0.19760 s ← **LOWER**
- OLS_LinearReg_Standardization: 0.20648 s

### **Final Decision**
**Winner: OLS_LinearReg_Normalization** 

**Justification:** Both runs achieve identical predictive performance on test data (MAE, RMSE, R²), as expected for OLS predictions which are mathematically scale-invariant. OLS_LinearReg_Normalization wins via the fit-time tiebreaker (0.19760 s vs 0.20648 s), providing marginally faster training while maintaining identical generalization performance. This demonstrates that MinMaxScaler produces slightly faster training than StandardScaler in this specific implementation.

### **Winner Hyperparameters**

| Parameter | Value |
|---|---|
| **fit_intercept** | True |

## **Overfitting/Underfitting Diagnosis (Evidence-Based Analysis)**

### **Train vs Test Gap Analysis**

| Metric | Train | Test | Gap | Interpretation |
|---|---:|---:|---:|---|
| **MAE** | 0.39666 | 0.40386 | +0.00720 (+1.82%) | Minimal gap; test error is only 1.82% higher than train |
| **RMSE** | 0.69367 | 0.71129 | +0.01762 (+2.54%) | Minimal gap; test error increases by 2.54% |
| **R² Score** | 0.99413 | 0.99387 | −0.00026 (−0.03%) | Negligible degradation on test data |

### **Diagnosis: No Evidence of Overfitting or Underfitting**

**Rationale:**
- **Train-Test Gaps are Minimal**: The gaps in MAE (+1.82%), RMSE (+2.54%), and R² (−0.03%) are all extremely small, indicating the model generalizes very well to unseen data.
- **No Significant Performance Degradation**: A properly regularized model typically shows gaps of 5–15% or higher when overfitting is present. These runs show near-identical performance.
- **Excellent Fit Quality**: R² scores above 0.99 on both train and test partitions indicate the model explains over 99% of variance in both datasets.
- **No Underfitting Signal**: High R² and low errors on both partitions confirm the model is not too simplistic.

### **Conclusion**
The OLS Linear Regression model exhibits **excellent generalization behavior** with no signs of overfitting or underfitting. The model achieves consistent, near-identical performance on both training and test datasets, demonstrating robust predictive capability for the continuous `diabetes_risk_score` target.